# SCADA Wind Power – Feature Engineering Exploration

This notebook demonstrates the feature engineering pipeline, showing how each
transformation contributes to predictive power.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.feature_engineering import FeatureEngineer

sns.set_theme(style='darkgrid')
%matplotlib inline

In [ ]:
# Generate synthetic SCADA data for demonstration
np.random.seed(42)
n = 8760
timestamps = pd.date_range('2022-01-01', periods=n, freq='h')
wind_speed = np.random.weibull(2, n) * 8
rotor_speed = wind_speed * 1.5 + np.random.normal(0, 0.5, n)
pitch_angle = np.clip(np.random.normal(5, 3, n), 0, 30)
active_power = np.clip(0.5 * 1.225 * 7854 * wind_speed**3 / 1000 + np.random.normal(0, 30, n), 0, None)

df_raw = pd.DataFrame({
    'Datetime': timestamps,
    'Wind Speed (m/s)': wind_speed,
    'Rotor Speed (rpm)': rotor_speed,
    'Pitch Angle (deg)': pitch_angle,
    'LV ActivePower (kW)': active_power,
})
print(f'Raw dataset: {df_raw.shape}')
df_raw.head()

In [ ]:
engineer = FeatureEngineer(config_path='../config.yaml')

# Step 1 – Wind Power Density
df_wpd = engineer.derive_wind_power_density(df_raw)

plt.figure(figsize=(10, 5))
plt.scatter(df_wpd['Wind Speed (m/s)'], df_wpd['wind_power_density'],
            alpha=0.2, s=5, c='darkorange')
plt.xlabel('Wind Speed (m/s)')
plt.ylabel('Wind Power Density (W)')
plt.title('Wind Power Density  (P = 0.5 × ρ × A × v³)')
plt.tight_layout()
plt.show()

print('Correlation of wind_power_density with target:',
      df_wpd[['wind_power_density', 'LV ActivePower (kW)']].corr().iloc[0, 1].round(4))

In [ ]:
# Step 2 – Temporal features
df_temp = engineer.create_temporal_features(df_wpd, datetime_col='Datetime')

hourly_power = df_temp.groupby('hour')['LV ActivePower (kW)'].mean()
plt.figure(figsize=(10, 4))
hourly_power.plot(kind='bar', color='steelblue')
plt.xlabel('Hour of Day')
plt.ylabel('Average Power (kW)')
plt.title('Average Power by Hour of Day')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Step 3 – Rolling statistics
df_roll = engineer.create_rolling_statistics(df_temp)

roll_cols = [c for c in df_roll.columns if 'rolling' in c and 'Wind Speed' in c]
fig, axes = plt.subplots(len(roll_cols), 1, figsize=(14, 3 * len(roll_cols)), sharex=True)

for ax, col in zip(axes, roll_cols):
    ax.plot(df_roll.index[:500], df_roll[col].values[:500], linewidth=0.8)
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Step 4 – Aerodynamic features
df_aero = engineer.create_aerodynamic_features(df_roll)

aero_cols = ['wind_rotor_interaction', 'pitch_wind_interaction',
             'wind_speed_squared', 'wind_speed_cubed']
aero_cols = [c for c in aero_cols if c in df_aero.columns]

target = 'LV ActivePower (kW)'
corrs = {c: df_aero[[c, target]].corr().iloc[0, 1] for c in aero_cols}

print('Correlations with target:')
for name, val in sorted(corrs.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {name:<30}  {val:.4f}')

In [ ]:
# Full pipeline
df_full = engineer.engineer_features(df_raw, datetime_col='Datetime')
print(f'Features after full pipeline: {df_full.shape[1]}')

# Correlation heatmap of top features with target
numeric_df = df_full.select_dtypes(include=[np.number])
top_corr = numeric_df.corr()[target].abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_corr.drop(target, errors='ignore').plot(kind='barh', color='steelblue')
plt.xlabel('|Pearson Correlation| with Target')
plt.title('Top Feature Correlations with Active Power')
plt.tight_layout()
plt.show()